In [3]:
import pandas as pd

df = pd.read_csv(r"data.csv", sep=";", encoding="utf-8-sig")
df.columns = df.columns.str.strip()

n_rows, n_cols = df.shape
n_features = n_cols - 1  #last column is Target
print(f"Rows: {n_rows}, Features: {n_features}")
assert n_rows >= 500 and n_features >= 12, "Dataset does not meet assignment minimums"

print("\nMissing values:", df.isna().sum().sum())
print("\nClass distribution:")
print(df["Target"].value_counts())
print("\nDtypes:")
print(df.dtypes.value_counts())

Rows: 4424, Features: 36

Missing values: 0

Class distribution:
Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64

Dtypes:
int64      29
float64     7
str         1
Name: count, dtype: int64


In [4]:
import numpy as np
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef)

In [6]:
# random seed 
SEED = 4091  
# test size for train/test split        
TEST_SIZE = 0.2

In [7]:
# data split for testset 
train_df, test_df = train_test_split(
    df, test_size=TEST_SIZE, stratify=df["Target"], random_state=SEED
)
test_df.to_csv("test_data.csv", index=False)


In [8]:
print(len(train_df), len(test_df))                  
print(pd.read_csv("test_data.csv").shape)

3539 885
(885, 37)


In [9]:
#label encoding for target variable
le = LabelEncoder()
y_train = le.fit_transform(train_df["Target"])
y_test = le.transform(test_df["Target"])
X_train = train_df.drop(columns="Target")
X_test = test_df.drop(columns="Target")

In [11]:
#column groups
nominal_cols = [
    "Marital status", "Application mode", "Course", "Previous qualification",
    "Nacionality", "Mother's qualification", "Father's qualification",
    "Mother's occupation", "Father's occupation", "Daytime/evening attendance",
]
binary_cols = [
    "Displaced", "Educational special needs", "Debtor",
    "Tuition fees up to date", "Gender", "Scholarship holder", "International",
]
numeric_cols = [c for c in X_train.columns if c not in nominal_cols + binary_cols]


In [12]:
#data preprocessing pipeline
def make_preprocessor():
    return ColumnTransformer([
        ("nom", OneHotEncoder(handle_unknown="infrequent_if_exist",
                              min_frequency=20, sparse_output=False), nominal_cols),
        ("num", StandardScaler(), numeric_cols),
        ("bin", "passthrough", binary_cols),
    ])


In [13]:
#models 
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=SEED),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, min_samples_leaf=5, random_state=SEED),
    "KNN": KNeighborsClassifier(n_neighbors=9),
    "Naive Bayes": GaussianNB(),
    "Support Vector Machine": CalibratedClassifierCV(SVC(kernel="rbf", C=1.0, probability=True, random_state=SEED), cv=5),
}


In [ ]:

#training and evaluation
Path("model").mkdir(exist_ok=True)
rows = []
for name, clf in models.items():
    pipe = Pipeline([("prep", make_preprocessor()), ("clf", clf)])
    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)

    rows.append({
        "ML Model Name": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro"),
        "Precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Recall": recall_score(y_test, y_pred, average="macro"),
        "F1": f1_score(y_test, y_pred, average="macro"),
        "MCC": matthews_corrcoef(y_test, y_pred),
    })
    joblib.dump(pipe, f"model/{name.lower().replace(' ', '_')}.joblib")

joblib.dump(le, "model/label_encoder.joblib")

results = pd.DataFrame(rows).set_index("ML Model Name").round(4)
results.to_csv("results.csv")
print(results)

c:\Users\mamid\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\mamid\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\mamid\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\mamid\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\svm\

                        Accuracy     AUC  Precision  Recall      F1     MCC
ML Model Name                                                              
Logistic Regression       0.8011  0.9106     0.7676  0.7230  0.7360  0.6708
Decision Tree             0.7605  0.8479     0.7085  0.6806  0.6899  0.6028
KNN                       0.7130  0.8254     0.6585  0.6084  0.6172  0.5194
Naive Bayes               0.4294  0.7466     0.5793  0.5061  0.4391  0.2822
Support Vector Machine    0.7932  0.8686     0.7580  0.6958  0.7077  0.6581
